In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/rq3_inputs.csv', index_col='Datetime (UTC)', parse_dates=True)

In [3]:
BATTERY_CAPACITY = 500
MIN_SOC = 0.1 * BATTERY_CAPACITY
MAX_SOC = 0.9 * BATTERY_CAPACITY
MAX_CHARGE_RATE = 100
MAX_DISCHARGE_RATE = 100
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95

In [4]:
PRICE_THRESHOLD_HIGH = df['predicted_price'].quantile(0.75)
PRICE_THRESHOLD_LOW = df['predicted_price'].quantile(0.25)

print(f"Праг за продажба: {PRICE_THRESHOLD_HIGH:.2f} EUR/MWh")
print(f"Праг за складирање: {PRICE_THRESHOLD_LOW:.2f} EUR/MWh")

Праг за продажба: 115.02 EUR/MWh
Праг за складирање: 78.77 EUR/MWh


In [5]:
def rule_based_decision(row):
    if row['predicted_price'] > PRICE_THRESHOLD_HIGH:
        return 'sell'
    elif row['predicted_price'] < PRICE_THRESHOLD_LOW:
        return 'store'
    else:
        return 'hold'

df['action_rule_based'] = df.apply(rule_based_decision, axis=1)
print(df['action_rule_based'].value_counts())

action_rule_based
hold     152
sell      77
store     77
Name: count, dtype: int64


In [6]:
soc = MIN_SOC
revenue_rule_based = 0
soc_history = []

for idx, row in df.iterrows():
    actual_price = row['actual_price']
    actual_production = row['actual_P']
    action = row['action_rule_based']

    if action == 'sell':
        revenue_rule_based += actual_production * actual_price / 1000
        discharge_amount = min(MAX_DISCHARGE_RATE, soc - MIN_SOC)
        if discharge_amount > 0:
            soc -= discharge_amount
            revenue_rule_based += discharge_amount * DISCHARGE_EFFICIENCY * actual_price / 1000

    elif action == 'store':
        charge_amount = min(actual_production, MAX_CHARGE_RATE, MAX_SOC - soc)
        soc += charge_amount * CHARGE_EFFICIENCY
        leftover = actual_production - charge_amount
        if leftover > 0:
            revenue_rule_based += leftover * actual_price / 1000

    else:  # hold
        revenue_rule_based += actual_production * actual_price / 1000

    soc_history.append(soc)

df['battery_soc'] = soc_history
print(f"Вкупен приход (Rule-based, со discharge): {revenue_rule_based:.2f} EUR")
print(f"Финална состојба: {soc:.2f} kWh")

Вкупен приход (Rule-based, со discharge): 1027.61 EUR
Финална состојба: 50.00 kWh


In [7]:
print(f"Финална состојба на батерија (SoC): {soc:.2f} kWh")
print(f"Максимален SoC достигнат: {max(soc_history):.2f} kWh")
print(f"Минимален SoC достигнат: {min(soc_history):.2f} kWh")

Финална состојба на батерија (SoC): 50.00 kWh
Максимален SoC достигнат: 450.00 kWh
Минимален SoC достигнат: 50.00 kWh
